# App 1 — Market Basket Analysis với AprioriFIM

Notebook trình diễn và tái hiện thực nghiệm Market Basket Analysis trên bộ dữ liệu **Groceries**. Notebook chỉ điều phối việc nạp dữ liệu và gọi lại các hàm đã có trong package `AprioriFIM` cùng ứng dụng `applications/market_basket.jl`; **không cài đặt lại thuật toán Apriori**.

**Cách chạy:** mở notebook bằng Google Colab (không cần đăng nhập GitHub), chọn *Runtime → Run all*. Dùng Python runtime mặc định. Cell thiết lập sẽ tự cài Julia qua `juliacall`, clone nhánh `main` (có kèm dữ liệu Groceries) và kích hoạt project. Lần chạy đầu cần Internet và có thể mất vài phút để tải Julia.

## 1. Giới thiệu bài toán

Market Basket Analysis tìm các nhóm sản phẩm thường xuất hiện cùng nhau trong một giao dịch. Quy trình demo:

**Dữ liệu giao dịch → Frequent Itemset Mining → Sinh luật kết hợp → Lọc/xếp hạng → Diễn giải kinh doanh.**

Một luật có dạng `X ⇒ Y`, trong đó hai vế không giao nhau:

- **Support:** tỷ lệ giao dịch chứa đồng thời `X` và `Y`.
- **Confidence:** tỷ lệ giao dịch chứa `Y` trong số các giao dịch đã chứa `X`.
- **Lift:** so sánh Confidence với mức phổ biến nền của `Y`. `Lift > 1` cho thấy hai vế đồng xuất hiện nhiều hơn mức kỳ vọng nếu độc lập.

Luật kết hợp mô tả mối liên hệ thống kê, không chứng minh quan hệ nhân quả.

## 2. Thiết lập môi trường Julia và nạp package

Notebook clone trực tiếp nhánh `main` của repository vì nhánh này chứa đầy đủ mã App 1 và `data/market/groceries.csv`. Không ghim commit `f965de2` thuộc nhánh `feat/chapter3-app1`, vì commit đó không có file dữ liệu được Git theo dõi.

Mỗi lần chạy, Notebook in ra SHA đầy đủ của commit `main` thực tế được tải để người dùng ghi nhận phiên bản mã nguồn. Julia được cài tự động qua `juliacall`, kích hoạt `Project.toml` và nạp `AprioriFIM`. Notebook không cài đặt lại thuật toán Apriori.

In [1]:
import os, sys, subprocess, pathlib, hashlib, shutil
from IPython.display import display, Markdown
import pandas as pd

REPO_URL = "https://github.com/TaVanDuc2005/FIM_Group14.git"
REPO_BRANCH = "main"
REPO_DIR = pathlib.Path("/content/FIM_Group14")

# Luôn lấy đúng nhánh main, nơi có cả mã App 1 và groceries.csv.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
    REPO_URL, str(REPO_DIR)
], check=True)

required = [
    REPO_DIR / "Project.toml",
    REPO_DIR / "src" / "AprioriFIM.jl",
    REPO_DIR / "applications" / "market_basket.jl",
    REPO_DIR / "data" / "market" / "groceries.csv",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Nhánh main thiếu tệp cần thiết: " + ", ".join(missing))

REPO_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Repository:", REPO_DIR)
print("Branch:", REPO_BRANCH)
print("Resolved commit:", REPO_COMMIT)

# juliacall tự tải/cài Julia nếu runtime chưa có Julia.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "juliacall"], check=True)
from juliacall import Main as jl

repo_julia = str(REPO_DIR).replace("\\", "/")
jl.seval(f'''
using Pkg
Pkg.activate(raw"{repo_julia}")
using AprioriFIM
include(raw"{repo_julia}/applications/market_basket.jl")
println("Julia version: ", VERSION)
println("Project: ", Base.active_project())
println("AprioriFIM loaded successfully.")
''')


Repository: /content/FIM_Group14
Branch: main
Resolved commit: 8c35dc43f845e73fd554aeafa31e24799cbaa6a0
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliacall/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] WARNING: You have Julia 1.12.6 installed but 1.10.3 - 1.11 is required.
[juliapkg]   It is recommended that you upgrade Julia or install JuliaUp.
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.9 to /root/.julia/environments/pyjuliapkg/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Dow

  Activating project at `/content/FIM_Group14`
[ Info: Precompiling AprioriFIM [5f544ea2-1aa2-4c25-b8c8-b024a6782403] 


Julia version: 1.11.9
Project: /content/FIM_Group14/Project.toml
AprioriFIM loaded successfully.


## 3. Đọc dữ liệu Groceries và kiểm tra thống kê

Dữ liệu `data/market/groceries.csv` được lấy trực tiếp từ nhánh `main` của repository. Notebook kiểm tra SHA-256 theo quy tắc chuẩn hóa CRLF/LF và dùng `load_groceries` của App 1, trong đó gọi `read_labeled_transactions` và `encode_transactions` của package.

In [2]:
data_path = REPO_DIR / "data" / "market" / "groceries.csv"
expected_sha256 = "f8a7074231acf5f3de6bdb7ec63fed03c28745d76bd49aa8ddc0c5d1cfa3da91"

# Chuẩn hóa CRLF về LF chỉ để kiểm tra checksum; không sửa file gốc.
normalized = b"\n".join(line.rstrip(b"\r") for line in data_path.read_bytes().splitlines()) + b"\n"
actual_sha256 = hashlib.sha256(normalized).hexdigest()
if actual_sha256 != expected_sha256:
    raise ValueError(f"Checksum dữ liệu không khớp: {actual_sha256}")

data_path_julia = str(data_path).replace("\\", "/")
jl.seval(f'transactions, labels = load_groceries(raw"{data_path_julia}")')

n_transactions = int(jl.seval("length(transactions)"))
n_items = int(jl.seval("length(labels)"))
basket_sizes = list(jl.seval("[length(t) for t in transactions]"))
total_item_occurrences = sum(basket_sizes)

stats = pd.DataFrame([
    ("Số giao dịch", n_transactions),
    ("Số mặt hàng phân biệt", n_items),
    ("Tổng lượt xuất hiện mặt hàng", total_item_occurrences),
    ("Kích thước giỏ trung bình", round(sum(basket_sizes) / n_transactions, 3)),
    ("Kích thước giỏ lớn nhất", max(basket_sizes)),
    ("Giỏ chỉ có 1 mặt hàng", sum(x == 1 for x in basket_sizes)),
    ("SHA-256", actual_sha256),
], columns=["Chỉ số", "Giá trị"])
display(stats)

,Chỉ số,Giá trị
0,Số giao dịch,9835
1,Số mặt hàng phân biệt,169
2,Tổng lượt xuất hiện mặt hàng,43367
3,Kích thước giỏ trung bình,4.409
4,Kích thước giỏ lớn nhất,32
5,Giỏ chỉ có 1 mặt hàng,2159
6,SHA-256,f8a7074231acf5f3de6bdb7ec63fed03c28745d76bd49a...


## 4. Khảo sát tham số — 35 tổ hợp `minsup × minconf`

Gọi trực tiếp `sweep_thresholds(transactions)` trong `applications/market_basket.jl`. Hàm thực hiện 7 mức `minsup` × 5 mức `minconf`, tổng cộng **35 tổ hợp**. Với mỗi mức `minsup`, Apriori chỉ chạy một lần; các mức `minconf` được khảo sát trên kết quả tập phổ biến đó.

Các cột gồm số tập phổ biến, số luật thô, số luật có `Lift > 1`, số luật sau tinh gọn, support nhỏ nhất của vế kết luận và thống kê Lift.

In [3]:
jl.seval(r'''
sweep_results = sweep_thresholds(transactions)
open("/content/sweep_results.tsv", "w") do io
    println(io, join(["minsup","minsup_count","minconf","n_itemsets","n_rules",
                      "n_lift_gt1","n_curated","min_consequent_count",
                      "median_lift","max_lift"], '\t'))
    for r in sweep_results
        println(io, join([r.minsup, r.minsup_count, r.minconf, r.n_itemsets,
                          r.n_rules, r.n_lift_gt1, r.n_curated,
                          r.min_consequent_count, r.median_lift, r.max_lift], '\t'))
    end
end
''')
sweep_df = pd.read_csv("/content/sweep_results.tsv", sep="\t")
assert len(sweep_df) == 35, f"Mong đợi 35 tổ hợp, nhận {len(sweep_df)}"
display(sweep_df.style.format({
    "minsup": "{:.3f}", "minconf": "{:.1f}",
    "median_lift": "{:.3f}", "max_lift": "{:.3f}"
}))

,minsup,minsup_count,minconf,n_itemsets,n_rules,n_lift_gt1,n_curated,min_consequent_count,median_lift,max_lift
0,0.050,492.000000,0.2,31.000000,6.000000,6.000000,3.000000,1372.000000,1.514,1.572
1,0.050,492.000000,0.3,31.000000,3.000000,3.000000,3.000000,2513.000000,1.514,1.572
2,0.050,492.000000,0.4,31.000000,1.000000,1.000000,1.000000,2513.000000,1.572,1.572
3,0.050,492.000000,0.5,31.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000
4,0.050,492.000000,0.6,31.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000
5,0.020,197.000000,0.2,122.000000,73.000000,70.000000,57.000000,736.000000,1.557,2.842
6,0.020,197.000000,0.3,122.000000,37.000000,37.000000,34.000000,1072.000000,1.640,2.842
7,0.020,197.000000,0.4,122.000000,15.000000,15.000000,14.000000,1903.000000,1.805,2.450
8,0.020,197.000000,0.5,122.000000,1.000000,1.000000,1.000000,2513.000000,2.007,2.007
9,0.020,197.000000,0.6,122.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.000


## 5. Khai phá luật với ngưỡng được chọn

Theo báo cáo thực nghiệm, cấu hình được chọn là `minsup = 0.005` (support count tối thiểu 50 giao dịch) và `minconf = 0.3`.

Notebook gọi `run_market_basket` để chạy Apriori và sinh luật kết hợp. Đây là pipeline có sẵn trong App 1, không có bản cài đặt thuật toán riêng trong Notebook.

In [4]:
jl.seval("""
selected_minsup = 0.005
selected_minconf = 0.3
itemsets, rules = run_market_basket(
    transactions; minsup=selected_minsup, minconf=selected_minconf
)
""")
n_itemsets = int(jl.seval("length(itemsets)"))
n_rules = int(jl.seval("length(rules)"))
print(f"minsup = 0.005 | minconf = 0.3")
print(f"Số tập phổ biến: {n_itemsets:,}")
print(f"Số luật kết hợp sinh ra: {n_rules:,}")

minsup = 0.005 | minconf = 0.3
Số tập phổ biến: 1,001
Số luật kết hợp sinh ra: 487


## 6. Lọc, tinh gọn và xếp hạng luật

Thứ tự xử lý bám theo App 1:

1. Giữ các luật có `Lift > 1` bằng `curate_rules`.
2. Loại luật dư thừa theo heuristic đã cài trong ứng dụng: cùng vế phải nhưng có luật tổng quát hơn với Confidence không thấp hơn; đồng thời chỉ giữ một luật tốt nhất cho mỗi itemset.
3. Lấy Top 10 theo Lift và Top 10 theo Support bằng `top_rules`.

Support trong package được lưu dưới dạng **số giao dịch tuyệt đối**; cột Support (%) dưới đây được tính bằng `support / 9835 × 100`.

In [5]:
jl.seval(r'''
curated_rules = curate_rules(rules)

function export_top_rules(path, selected, labels, n_tx)
    open(path, "w") do io
        println(io, join(["Rank","Rule","Support_count","Support_pct","Confidence","Lift"], '\t'))
        for (i, r) in enumerate(selected)
            rule_text = format_rule(r, labels; digits=3)
            println(io, join([i, rule_text, r.support,
                              100 * r.support / n_tx,
                              r.confidence, r.lift], '\t'))
        end
    end
end

top_lift = top_rules(curated_rules, 10; by=:lift)
top_support = top_rules(curated_rules, 10; by=:support)
export_top_rules("/content/top_lift.tsv", top_lift, labels, length(transactions))
export_top_rules("/content/top_support.tsv", top_support, labels, length(transactions))
''')

n_curated = int(jl.seval("length(curated_rules)"))
print(f"Số luật Lift > 1 và sau tinh gọn: {n_curated:,}")
top_lift_df = pd.read_csv("/content/top_lift.tsv", sep="\t")
top_support_df = pd.read_csv("/content/top_support.tsv", sep="\t")

display(Markdown("### Top 10 luật theo Lift"))
display(top_lift_df.style.format({
    "Support_pct": "{:.2f}%", "Confidence": "{:.3f}", "Lift": "{:.3f}"
}))
display(Markdown("### Top 10 luật theo Support"))
display(top_support_df.style.format({
    "Support_pct": "{:.2f}%", "Confidence": "{:.3f}", "Lift": "{:.3f}"
}))

Số luật Lift > 1 và sau tinh gọn: 348


### Top 10 luật theo Lift

,Rank,Rule,Support_count,Support_pct,Confidence,Lift
0,1,"{pip fruit, root vegetables} => {other vegetables, whole milk} sup = 54, conf = 0.353, lift = 4.716",54,0.55%,0.353,4.716
1,2,"{root vegetables, tropical fruit} => {other vegetables, whole milk} sup = 69, conf = 0.333, lift = 4.454",69,0.70%,0.333,4.454
2,3,"{citrus fruit, root vegetables} => {other vegetables, whole milk} sup = 57, conf = 0.328, lift = 4.377",57,0.58%,0.328,4.377
3,4,"{root vegetables, whipped/sour cream} => {other vegetables, whole milk} sup = 51, conf = 0.304, lift = 4.057",51,0.52%,0.304,4.057
4,5,"{root vegetables, yogurt} => {other vegetables, whole milk} sup = 77, conf = 0.303, lift = 4.051",77,0.78%,0.303,4.051
5,6,"{herbs} => {root vegetables} sup = 69, conf = 0.431, lift = 3.956",69,0.70%,0.431,3.956
6,7,"{citrus fruit, pip fruit} => {tropical fruit} sup = 55, conf = 0.404, lift = 3.854",55,0.56%,0.404,3.854
7,8,"{root vegetables, whole milk, yogurt} => {tropical fruit} sup = 56, conf = 0.392, lift = 3.732",56,0.57%,0.392,3.732
8,9,"{curd, tropical fruit} => {yogurt} sup = 52, conf = 0.515, lift = 3.691",52,0.53%,0.515,3.691
9,10,"{beef, other vegetables} => {root vegetables} sup = 78, conf = 0.402, lift = 3.689",78,0.79%,0.402,3.689


### Top 10 luật theo Support

,Rank,Rule,Support_count,Support_pct,Confidence,Lift
0,1,"{other vegetables} => {whole milk} sup = 736, conf = 0.387, lift = 1.514",736,7.48%,0.387,1.514
1,2,"{rolls/buns} => {whole milk} sup = 557, conf = 0.308, lift = 1.205",557,5.66%,0.308,1.205
2,3,"{yogurt} => {whole milk} sup = 551, conf = 0.402, lift = 1.572",551,5.60%,0.402,1.572
3,4,"{root vegetables} => {whole milk} sup = 481, conf = 0.449, lift = 1.756",481,4.89%,0.449,1.756
4,5,"{root vegetables} => {other vegetables} sup = 466, conf = 0.435, lift = 2.247",466,4.74%,0.435,2.247
5,6,"{yogurt} => {other vegetables} sup = 427, conf = 0.311, lift = 1.608",427,4.34%,0.311,1.608
6,7,"{tropical fruit} => {whole milk} sup = 416, conf = 0.403, lift = 1.578",416,4.23%,0.403,1.578
7,8,"{tropical fruit} => {other vegetables} sup = 353, conf = 0.342, lift = 1.768",353,3.59%,0.342,1.768
8,9,"{bottled water} => {whole milk} sup = 338, conf = 0.311, lift = 1.217",338,3.44%,0.311,1.217
9,10,"{pastry} => {whole milk} sup = 327, conf = 0.374, lift = 1.463",327,3.32%,0.374,1.463


## 7. Phân tích kết quả và ý nghĩa kinh doanh

### 7.1. Các luật có Lift cao

Theo kết quả thực nghiệm trong báo cáo, nhóm luật nổi bật gồm:

- `{pip fruit, root vegetables} ⇒ {other vegetables, whole milk}`: Lift khoảng **4,716**, Support 54 giao dịch (0,55%). Đây là liên hệ nổi bật nhưng xuất hiện trong tỷ lệ nhỏ giao dịch.
- `{herbs} ⇒ {root vegetables}`: Confidence khoảng **0,431**, Lift khoảng **3,956**. Có thể thử nghiệm gợi ý mua kèm nhóm rau củ khi khách chọn herbs.
- `{curd, tropical fruit} ⇒ {yogurt}`: Confidence khoảng **0,515**, Lift khoảng **3,691**. Có thể thử nghiệm gợi ý yogurt hoặc coupon đi kèm.

### 7.2. Phân biệt Lift và Support

Luật `{other vegetables} ⇒ {whole milk}` đứng đầu theo Support trong báo cáo: 736 giao dịch (7,48%), nhưng Lift khoảng 1,514. Ngược lại, luật có Lift cao nhất chỉ xuất hiện trong 54 giao dịch (0,55%).

- **Support** phù hợp để nhận diện hành vi mua phổ biến, hỗ trợ dự trữ và bố trí hàng.
- **Lift** giúp tìm các tổ hợp có mức liên kết nổi bật so với xác suất nền, phù hợp để đề xuất mua kèm hoặc thử nghiệm combo.
- Nên xem đồng thời cả hai chỉ số; Lift cao không tự động đồng nghĩa với quy mô kinh doanh lớn.

### 7.3. Lưu ý khi diễn giải

Các luật chỉ mô tả sự đồng xuất hiện trong dữ liệu quan sát. Chúng không chứng minh mua mặt hàng ở vế trái gây ra việc mua mặt hàng ở vế phải. Những đề xuất như bố trí gần nhau, combo hoặc coupon cần được kiểm chứng bằng thử nghiệm kinh doanh có đối chứng. Kết quả cũng phụ thuộc vào ngưỡng và heuristic tinh gọn được chọn.

## 8. Kiểm tra kết quả và tái hiện

Cell dưới đây đối chiếu các con số chính với kết quả đã trình bày trong báo cáo App 1. Nếu mã nguồn hoặc dữ liệu thay đổi, Notebook sẽ báo rõ thay vì âm thầm trình bày kết quả cũ.

In [6]:
expected = {
    "transactions": 9835,
    "items": 169,
    "itemsets": 1001,
    "rules": 487,
    "curated": 348,
}
observed = {
    "transactions": n_transactions,
    "items": n_items,
    "itemsets": n_itemsets,
    "rules": n_rules,
    "curated": n_curated,
}
comparison = pd.DataFrame([
    {"Chỉ số": k, "Kỳ vọng theo báo cáo": v, "Kết quả chạy Notebook": observed[k],
     "Khớp": v == observed[k]}
    for k, v in expected.items()
])
display(comparison)
if not comparison["Khớp"].all():
    print("LƯU Ý: Có chỉ số không khớp báo cáo. Hãy kiểm tra phiên bản mã nguồn, dữ liệu và môi trường.")
else:
    print("Tất cả chỉ số kiểm tra chính khớp với báo cáo App 1.")

,Chỉ số,Kỳ vọng theo báo cáo,Kết quả chạy Notebook,Khớp
0,transactions,9835,9835,True
1,items,169,169,True
2,itemsets,1001,1001,True
3,rules,487,487,True
4,curated,348,348,True


Tất cả chỉ số kiểm tra chính khớp với báo cáo App 1.


---

## Kết luận

Notebook là lớp trình diễn có thể chạy lại trên Google Colab. Phần khai phá vẫn được thực hiện bởi package `AprioriFIM` và pipeline App 1 có sẵn trong repository; Notebook không chứa một bản triển khai Apriori thứ hai.

**Nguồn mã:** [FIM_Group14 trên GitHub](https://github.com/TaVanDuc2005/FIM_Group14)  
**Nguồn dữ liệu:** Groceries dataset; mô tả và đường dẫn tải được ghi trong `data/market/README.md` của repository.